## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **6fe90fe9-6a77-4aa6-9736-7872aad47731**


## Dataset

# Agglo - Carroyage de population et emplois


## Description

Selon l&#x27;Institut National de la Statistique et des Etudes Economiques &#40;INSEE&#41;, un carroyage est un découpage de l&#x27;espace géographique en mailles régulières de forme carrée et de taille fixe. Construits sans a priori sur ce que doivent être les zonages d&#x27;intervention ou de gestion, les carroyages permettent une vision continue des phénomènes et sont naturellement à même de mettre en évidence les zones à enjeu. Depuis 2021, le Grand Genève est doté d&#x27;une grille homogène et continue de 166&#x27;792 carreaux de 200 m &#40;dont 55&#x27;438 sont situés au sein du Grand Genève&#41;. Le carroyage Grand Genève permet notamment une lecture localisée de la densité &#40;habitants ou emplois / ha&#41; et mixité fonctionnelle &#40;part de l&#x27;emploi sur la somme des emplois et des habitants&#41; ou d&#x27;autres analyses localisées à l&#x27;échelle infra communale. Par contre, cette donnée n&#x27;est pas indiquée pour la réalisation d&#x27;agrégations par divisions administratives transfrontalières, ni pour des comparaisons entre différents millésimes, pour lesquelles les statistiques de l&#x27;Observatoire Statistique Transfrontalier &#40;OST, https://www.ge.ch/statregio-francosuisse/&#41; constituent la référence. Les données carroyées de population et d&#x27;emplois du Grand Genève sont construites à partir des données INSEE et OFS. Les sources sont les suivantes: Suisse - population: STAPOP, 2021; - emplois: STATENT, 2021. France - population: FiLoSoFi, 2021; - emplois: estimation réalisée par le Pôle métropolitain du Genevois Français sur la base des données SIRENE, 2020. Cette base localise à l&#x27;adresse et classe les entreprises en fonction de leur taille &#40;rangs de nombre d&#x27;employés&#41;. Le millésime 2021 retenu pour la population correspond à la dernière version disponible pour la France et permet donc des comparaisons transfrontalières en matière de densités d&#x27;habitants. Par contre, les données d&#x27;entreprises géoréférencés en France &#40;SIRENE&#41; ne sont pas disponibles de manière suffisamment fiable avant 2020. Il est prévu d&#x27;actualiser les données de population et d&#x27;emplois dès que le prochain millésime du carreau de population sera mis à disposition par l&#x27;INSEE.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/agglo-carroyage-de-population-et-emplois1)


## Metadata

- **identifier:** AGGLO&#95;CARREAU&#95;200@canton-geneve
- **publisher.name:** Direction du projet d&#x27;agglomération Grand Genève
- **organization.display&#95;name:** Kanton Genf
- **organization.url:** 
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** population-emploi-insee-carreau-statpop-statent-statistique
- **issued:** 2025-06-12 00:00:00
- **metadata&#95;created:** 2026-01-12T16:01:17.858882
- **metadata&#95;modified:** 2026-11-12T03:45:42.887553
- **Resource id:** 2c95f4ce-1a42-422b-a3ae-5a1df542c615
- **Resource name:** CSV
- **Resource package&#95;id:** 6fe90fe9-6a77-4aa6-9736-7872aad47731
- **Resource description:** Service téléchargement extracteur
- **Resource issued:** 2025-06-12 00:00:00
- **Resource modified:** 2026-10-29 00:00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;open


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **CSV** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://extract.sitg.ge.ch/files/AGGLO_CARREAU_200-csv.zip")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** Annina.Hirschi-Wyss@etat.ge.ch &#124; Direction du projet d&#x27;agglomération Grand Genève
